# CwH alpha sweep — Stage 1: causal FEATURE TABLE + single-factor tables (PIT_C base)

**Base:** PIT_C detector (causal-min handle), v2b 50>200 filter, t05 exit sim, per-ticker
non-overlap — code copied verbatim from `cup_handle_15yr_history_PIT_C.ipynb`; parity is
asserted against that notebook's saved outputs.

**LOCKBOX (enforced, not promised):** every price series (corpus, SPY, RSP) is CUT at
2024-12-31 inside the loader — no 2025+ bar is ever kept in memory. IS = entries
<= **2024-06-28** (purge: the 120-bar time stop resolves every IS trade before the cut).
Nothing dated after the purge is tabulated, printed or saved.
Prior exposure (recorded in the trial log): the PIT_C base and hscore quintiles were already
evaluated on 2025+ in the PIT_C notebook. The lockbox is clean for every feature below.

**Causality:** every feature uses bars <= the fire bar `j` (the close above the rim), except
`gap_pct` (timing=entry, uses the entry-bar OPEN only). Cell 6 re-derives every feature on data
CUT at `j` (and at `j+1` with only the open known, for gap + geometry) and aborts on any mismatch;
a deliberately leaky negative control must FAIL or the tester is declared blind.

**Pass criteria (fixed before any table; printed with results):**
PF after 0.05R round-trip cost >= 1.30 · month-block bootstrap 90% CI lower bound on PF (after cost) > 1.0 ·
n >= 300 · rule avg R beats the baseline avg R in >= 10 of 14 IS years · no single year > 25% of total net R (after cost).

**Trials:** per continuous feature, IS-quintile buckets -> 13 rules (B1..B5, GE2..GE5, LE1..LE4), both
directions; binary features -> EQ1/EQ0. Every rule is appended to `sweep_log_PIT_C.csv` (DSR inputs)
and its monthly R to `sweep_returns_PIT_C.csv` (PBO matrix); identical configs are not double-counted.
`prereg_PIT_C.json` is written before any table and is immutable thereafter.

Run top to bottom. Cells 2–4 are the slow part (~20–40 min).


In [ ]:
# ===== Cell 0 — Config + data checks (prints only) =====
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, glob, time, json, hashlib, os
from pathlib import Path
from numba import njit
import warnings; warnings.filterwarnings('ignore')

DATA_DIR   = Path('/content/drive/MyDrive/Bukowski')
HIST_DIR   = DATA_DIR/'results'/'history_15yr'      # READ-ONLY: PIT_C outputs, used for parity asserts
OUT_DIR    = DATA_DIR/'results'/'sweep_PIT_C'       # everything this notebook writes goes here
OUT_DIR.mkdir(parents=True, exist_ok=True)
REF_DIR    = DATA_DIR/'reference'                   # RSP.csv lives here, OUTSIDE the scan universe glob

# ---- LOCKBOX ----
CUT_DATE   = pd.Timestamp('2024-12-31')   # all prices cut here at load
PURGE_DATE = pd.Timestamp('2024-06-28')   # IS = entries <= this

# ---- constants identical to PIT_C ----
ATR_PERIOD, MAX_STOP_PCT, MIN_STOP_ATR = 14, 0.10, 1.0
STOCK_FAST_SMA, STOCK_SLOW_SMA = 50, 200
TIME_STOP_DAYS = 120
T05_TARGET_MULT = 0.5
CONFIRM_WINDOW_BARS = 15
DET = dict(pivot_left=5, pivot_right=5, min_cup_duration=30, max_cup_duration=130,
           min_cup_depth_pct=0.12, max_cup_depth_pct=0.50, rch_lch_tolerance=0.15,
           min_rounded_bars=3, rounded_band_pct=0.05, rounded_span_min=5,
           max_handle_duration=30, min_handle_retr_pct=0.10, max_handle_retr_pct=0.50)

# ---- PASS CRITERIA — fixed here, before any table ----
COST_R = 0.05
CRITERIA = dict(min_pf_cost=1.30, min_ci_lo=1.0, ci_level=0.90, boot_reps=2000,
                min_n=300, min_years_beat=10, n_is_years_expected=14, max_year_share=0.25)
SWEEP_ID = 'PIT_C_single_factor_v1'
BUILT_FROM_COMMIT = 'a767ff5595ba7efe65f8f63c0bf446dbd7d90593'

# ---- data checks ----
files = sorted(f for f in glob.glob(f'{DATA_DIR}/*.csv') if Path(f).stem.upper() != 'SPY')
print(f'{len(files)} ticker files')
vrows = []
for f in files:
    tk = Path(f).stem.upper()
    cols = pd.read_csv(f, nrows=0).columns
    if 'Volume' not in cols:
        vrows.append(dict(ticker=tk, has_volume=False, rows=np.nan, n_nan=np.nan, n_zero=np.nan)); continue
    v = pd.read_csv(f, usecols=['Date','Volume']); v['Date'] = pd.to_datetime(v['Date'])
    v = v[v['Date'] <= CUT_DATE]                                   # LOCKBOX
    vrows.append(dict(ticker=tk, has_volume=True, rows=len(v),
                      n_nan=int(v['Volume'].isna().sum()), n_zero=int((v['Volume']==0).sum())))
vchk = pd.DataFrame(vrows); vchk['bad_frac'] = (vchk.n_nan + vchk.n_zero) / vchk.rows
vchk.to_csv(OUT_DIR/'volume_check_PIT_C.csv', index=False)
print(f'Volume column missing: {int((~vchk.has_volume).sum())} files')
print(f'tickers with any NaN volume: {int((vchk.n_nan>0).sum())} | any zero volume: {int((vchk.n_zero>0).sum())}')
print(f'total NaN bars: {int(vchk.n_nan.sum())} | total zero bars: {int(vchk.n_zero.sum())}')
print('worst 15 by (NaN+zero)/rows:')
print(vchk.sort_values('bad_frac', ascending=False).head(15).to_string(index=False))

# volume split-adjustment check: median V 20 bars after / 20 before a known split.
# ~factor -> volume UNADJUSTED (prices are adjusted) ; ~1 -> volume split-adjusted too
SPLITS = [("NVDA","2024-06-10",10),("NVDA","2021-07-20",4),("AAPL","2020-08-31",4),("AAPL","2014-06-09",7),
          ("TSLA","2020-08-31",5),("TSLA","2022-08-25",3),("AMZN","2022-06-06",20),("GOOGL","2022-07-18",20),
          ("WMT","2024-02-26",3),("CMG","2024-06-26",50),("AVGO","2024-07-15",10),("SMCI","2024-10-01",10)]
print('\nvolume around known splits (after/before median ratio):')
for tk, day, fac in SPLITS:
    f = DATA_DIR/f'{tk}.csv'
    if not f.exists() or 'Volume' not in pd.read_csv(f, nrows=0).columns:
        print(f'  {tk:6s} {day}  n/a'); continue
    v = pd.read_csv(f, usecols=['Date','Volume']); v['Date'] = pd.to_datetime(v['Date'])
    v = v[v['Date'] <= CUT_DATE].sort_values('Date').reset_index(drop=True)
    i = v.index[v['Date'] >= day]
    if len(i)==0 or i[0] < 20: print(f'  {tk:6s} {day}  date n/a'); continue
    i = i[0]; ratio = v.Volume.iloc[i:i+20].median() / v.Volume.iloc[i-20:i].median()
    verdict = 'UNADJ' if abs(ratio-fac)/fac < 0.5 else ('ADJ' if 0.4 < ratio < 2.5 else '??')
    print(f'  {tk:6s} {day}  factor {fac:>2}  ratio {ratio:7.2f}  -> {verdict}')

RSP_PATH = DATA_DIR/'RSP.csv' if (DATA_DIR/'RSP.csv').exists() else REF_DIR/'RSP.csv'
print('\nSPY.csv present:', (DATA_DIR/'SPY.csv').exists(), '| RSP at', RSP_PATH, 'present:', RSP_PATH.exists())


In [ ]:
# ===== Cell 1 — RSP: one-off ADJUSTED Tiingo pull, only if missing =====
# Written to DATA_DIR/reference/RSP.csv — outside DATA_DIR/*.csv, so it never joins any scan universe.
# Cut at load like every other series (Cell 2). Prints row count only.
if RSP_PATH.exists():
    print('RSP present at', RSP_PATH, '— no pull')
else:
    import urllib.request
    token = os.environ.get('TIINGO_API_KEY')
    if not token:
        try:
            from google.colab import userdata; token = userdata.get('TIINGO_API_KEY')
        except Exception:
            token = None
    assert token, 'Set TIINGO_API_KEY (Colab secret or env) to pull RSP.'
    url = f'https://api.tiingo.com/tiingo/daily/RSP/prices?startDate=2003-01-01&token={token}'
    js = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=60)))
    rsp_raw = pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'),
                            'Open': js['adjOpen'], 'High': js['adjHigh'], 'Low': js['adjLow'],
                            'Close': js['adjClose'], 'Volume': js['adjVolume']})
    REF_DIR.mkdir(parents=True, exist_ok=True)
    rsp_raw.to_csv(RSP_PATH, index=False)
    print(f'RSP pulled: {len(rsp_raw)} rows (adjusted) -> {RSP_PATH}')
    del rsp_raw, js


In [ ]:
# ===== Cell 2 — Load prices, CUT AT LOAD =====
def compute_atr(df, period=14):
    h,l,c = df['High'],df['Low'],df['Close']; pc=c.shift(1)
    tr = pd.concat([(h-l),(h-pc).abs(),(l-pc).abs()],axis=1).max(axis=1)
    return tr.rolling(period).mean()

def load_px(path):
    """Read OHLC(+V), CUT at CUT_DATE before anything else touches it. Returns (cut_df, n_full_rows).
    n_full_rows (a row COUNT only) keeps the >=260-bar universe rule identical to PIT_C."""
    cols = pd.read_csv(path, nrows=0).columns
    use = ['Date','Open','High','Low','Close'] + (['Volume'] if 'Volume' in cols else [])
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date').reset_index(drop=True)
    n_full = len(d)
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d, n_full

def to_arrays(d):
    d = d.copy()
    d['ATR']  = compute_atr(d, ATR_PERIOD)
    d['S50']  = d['Close'].rolling(STOCK_FAST_SMA).mean()
    d['S200'] = d['Close'].rolling(STOCK_SLOW_SMA).mean()
    return dict(Date=d['Date'].values, Open=d['Open'].to_numpy(float), High=d['High'].to_numpy(float),
                Low=d['Low'].to_numpy(float), Close=d['Close'].to_numpy(float), Volume=d['Volume'].to_numpy(float),
                ATR=d['ATR'].to_numpy(float), S50=d['S50'].to_numpy(float), S200=d['S200'].to_numpy(float))

def read_is_only(path, cols):
    """Read only rows with entry_date <= PURGE_DATE (dates first, every other row skipped), so no
    post-purge row is ever loaded. Row order of the file is preserved."""
    dts = pd.to_datetime(pd.read_csv(path, usecols=['entry_date'])['entry_date'])
    keep = set((dts.index[dts <= PURGE_DATE] + 1).tolist())          # +1: header is line 0
    date_cols = [c for c in cols if c.endswith('_date')]
    return pd.read_csv(path, usecols=cols, parse_dates=date_cols,
                       skiprows=lambda i: i > 0 and i not in keep)[cols]
# ---- run ----
price_db = {}; t0 = time.time()
for i, f in enumerate(files):
    tk = Path(f).stem.upper()
    try:
        d, n_full = load_px(f)
        if n_full < 260 or len(d) == 0: continue
        price_db[tk] = to_arrays(d)
    except Exception:
        pass
    if (i+1) % 400 == 0: print(f'  loaded {len(price_db)} / {i+1} ({time.time()-t0:.0f}s)')
spy_df, _ = load_px(DATA_DIR/'SPY.csv'); rsp_df, _ = load_px(RSP_PATH)
SPY = dict(Date=spy_df['Date'].values, Close=spy_df['Close'].to_numpy(float))
RSP = dict(Date=rsp_df['Date'].values, Close=rsp_df['Close'].to_numpy(float))
assert max(v['Date'][-1] for v in price_db.values()) <= np.datetime64(CUT_DATE)
assert SPY['Date'][-1] <= np.datetime64(CUT_DATE) and RSP['Date'][-1] <= np.datetime64(CUT_DATE)
print(f'Universe: {len(price_db)} tickers | every series ends <= {CUT_DATE.date()} (asserted) | {time.time()-t0:.0f}s')


In [ ]:
# ===== Cell 3 — PIT_C detector (verbatim) + geometry fields =====
def find_pivot_highs(H, left=5, right=5):
    n=len(H); out=[]
    for i in range(left, n-right):
        w=H[i-left:i+right+1]
        if H[i]==w.max() and (w==H[i]).sum()==1: out.append(i)
    return out
def find_pivot_lows(L, left=5, right=5):
    n=len(L); out=[]
    for i in range(left, n-right):
        w=L[i-left:i+right+1]
        if L[i]==w.min() and (w==L[i]).sum()==1: out.append(i)
    return out

def detect_fires_full(tk, p, det):
    O,H,L,C,ATR,D = p['Open'],p['High'],p['Low'],p['Close'],p['ATR'],p['Date']
    n=len(C)
    ph = find_pivot_highs(H, det['pivot_left'], det['pivot_right'])
    pl = find_pivot_lows(L,  det['pivot_left'], det['pivot_right'])
    if len(ph) < 2: return []
    fires=[]
    for a,lch in enumerate(ph[:-1]):
        lch_high = H[lch]
        for rch in ph[a+1:]:
            dur = rch-lch
            if dur < det['min_cup_duration']: continue
            if dur > det['max_cup_duration']: break
            rch_high = H[rch]
            if abs(rch_high-lch_high)/lch_high > det['rch_lch_tolerance']: continue
            cup_lows = L[lch+1:rch]
            if len(cup_lows) < 5: continue
            cup_low = cup_lows.min(); cup_low_idx = lch+1+int(cup_lows.argmin())
            cup_rim = (lch_high+rch_high)/2
            cup_depth_pct = (cup_rim-cup_low)/cup_rim
            if not (det['min_cup_depth_pct'] <= cup_depth_pct <= det['max_cup_depth_pct']): continue
            band = cup_low*(1+det['rounded_band_pct'])
            full = L[lch:rch+1]; inb=[k for k,v in enumerate(full) if v<=band]
            if len(inb) < det['min_rounded_bars']: continue
            if (max(inb)-min(inb)) < det['rounded_span_min']: continue
            cup_depth_abs = cup_rim-cup_low
            if cup_depth_abs <= 0: continue
            breakout = max(lch_high, rch_high)
            # VARIANT C: breakout bar j = FIRST close > rim after the right rim. Only the
            # first counts — if it fails a check below, this cup gets no trade.
            j_lim = min(rch + 1 + det['max_handle_duration'] + CONFIRM_WINDOW_BARS, n)
            confirm=None
            for j in range(rch+1, j_lim):
                if C[j] > breakout: confirm=j; break
            if confirm is None: continue
            # causal-min handle: lowest low strictly between the right rim and j
            seg = L[rch+1:confirm]
            if len(seg) == 0: continue
            h_idx = rch+1+int(seg.argmin()); handle_low = L[h_idx]
            if h_idx - rch > det['max_handle_duration']: continue      # handle within 30 bars of rim
            if confirm - h_idx > CONFIRM_WINDOW_BARS: continue         # breakout within 15 bars of handle
            retr = (rch_high-handle_low)/cup_depth_abs
            if not (det['min_handle_retr_pct'] <= retr <= det['max_handle_retr_pct']): continue
            if handle_low <= cup_low: continue
            if confirm is None or confirm+1 >= n: continue      # need next-day open to enter
            atr = ATR[rch]
            if np.isnan(atr) or atr<=0: continue
            e_idx = confirm+1
            e_px  = O[e_idx]
            raw_stop = handle_low*0.999
            stop_dist = e_px-raw_stop
            if stop_dist <= 0: continue
            if stop_dist < MIN_STOP_ATR*atr: stop_dist = MIN_STOP_ATR*atr
            if stop_dist/e_px > MAX_STOP_PCT: continue
            fires.append(dict(
                ticker=tk, entry_idx=e_idx,
                entry_date=pd.Timestamp(D[e_idx]), entry_price=e_px,
                stop_price=e_px-stop_dist, target_price=breakout+cup_depth_abs,
                breakout_level=breakout, atr=atr,
                handle_low_date=pd.Timestamp(D[h_idx]), rch_date=pd.Timestamp(D[rch]),
                cup_depth_pct=round(cup_depth_pct*100,1), handle_retr_pct=round(retr*100,1),
                # === the 3 frozen handle_score features ===
                days_since_handle_low=int(e_idx-h_idx),
                handle_dur_days=int((pd.Timestamp(D[h_idx])-pd.Timestamp(D[rch])).days),
                handle_depth_atr=(breakout-handle_low)/atr,
                confirm_idx=int(confirm),
                # geometry indices/levels for the feature table (output-only additions)
                lch_idx=int(lch), rch_idx=int(rch), h_idx=int(h_idx), cup_low_idx=int(cup_low_idx),
                cup_low=float(cup_low), cup_rim=float(cup_rim), handle_low=float(handle_low)))
            break   # one cup per LCH (your logic)
    return fires

# ---- run ----
t0=time.time(); all_fires=[]
for i,(tk,p) in enumerate(price_db.items()):
    all_fires += detect_fires_full(tk,p,DET)
    if (i+1)%400==0: print(f'  scanned {i+1} tickers, {len(all_fires)} fires ({time.time()-t0:.0f}s)')
fires = pd.DataFrame(all_fires)
# TIE ALIGNMENT. PIT_C ran fires.sort_values('entry_date') — an UNSTABLE quicksort whose tie order
# depends on the array length — on the full 2010-2026 array, then kept the first row per
# (ticker, handle_low_date). Rows tied on (ticker, handle_low_date, entry_date) are different cups
# (different rch -> different ATR/rim -> different stop and R), so the pick matters. Our cut
# array is shorter, so its tie order differs. Reproduce PIT_C's pick from its saved raw-fire file
# (IS rows only; no outcomes in that file). Without the file, fall back to a deterministic order.
# Cups sharing (rch, handle low, breakout bar) can still differ by LEFT rim -> different cup low ->
# different target. PIT_C files carry no lch, but target_price (= rim + cup depth) separates them.
KEY = ['ticker','entry_date','handle_low_date','rch_date','_tgt']
def with_tgt(df): return df.assign(_tgt=df['target_price'].round(6))
fires = with_tgt(fires); fires['_pitc'] = False
raw_ref = HIST_DIR/'cup_handle_fires_15yr_raw_PIT_C.csv'
if raw_ref.exists():
    ref = with_tgt(read_is_only(raw_ref, KEY[:-1] + ['target_price']))
    fires['_pitc'] = pd.MultiIndex.from_frame(fires[KEY]).isin(pd.MultiIndex.from_frame(ref[KEY]))
else:
    print('WARNING: PIT_C raw-fire file not found — duplicate-cup ties resolved deterministically, parity NOT guaranteed')
fires = fires.sort_values(['entry_date','_pitc','ticker','lch_idx','rch_idx'],
                          ascending=[True, False, True, True, True], kind='mergesort')
fires = fires.drop_duplicates(['ticker','handle_low_date']).drop(columns='_pitc').reset_index(drop=True)
print(f'raw fires <= {CUT_DATE.date()}: {len(fires)} ({time.time()-t0:.0f}s)')


In [ ]:
# ===== Cell 4 — v2b + t05 sim + non-overlap (verbatim), purge, parity vs PIT_C =====
def uptrend_at(tk, e_idx):
    p = price_db[tk]; s50,s200 = p['S50'][e_idx], p['S200'][e_idx]
    if np.isnan(s50) or np.isnan(s200): return False
    return s50 > s200

@njit(cache=True)
def _sim_trade(opens,highs,lows,closes, entry_idx, entry_price, stop_price,
               target_price, atr, time_stop_days):
    n=len(opens); end=entry_idx+time_stop_days
    if end>n: end=n
    risk = entry_price-stop_price
    for i in range(entry_idx, end):
        bl=lows[i]; bh=highs[i]
        if bl <= stop_price:
            return i, stop_price, 0, (stop_price-entry_price)/risk
        if bh >= target_price:
            return i, target_price, 1, (target_price-entry_price)/risk
    last=end-1
    return last, closes[last], 4, (closes[last]-entry_price)/risk

# warm up
_sim_trade(np.array([100.,101,102]),np.array([100.,101,102]),np.array([99.,100,101]),
           np.array([100.,101,102]),0,100.,95.,110.,2.,120)

REASON={0:'stop',1:'target',4:'time_stop'}
# ---- run ----
fires['stock_uptrend'] = [uptrend_at(t,i) for t,i in zip(fires.ticker, fires.entry_idx)]
v2b = fires[fires['stock_uptrend']].reset_index(drop=True)
res = []
for f in v2b.itertuples():
    p = price_db[f.ticker]; ei = int(f.entry_idx)
    tgt = f.breakout_level + T05_TARGET_MULT*(f.target_price - f.breakout_level)   # t05 half target
    xi,xpx,rc,R = _sim_trade(p['Open'],p['High'],p['Low'],p['Close'],
                             ei, f.entry_price, f.stop_price, tgt, f.atr, TIME_STOP_DAYS)
    res.append(dict(exit_date=pd.Timestamp(p['Date'][xi]), exit=xpx, R=R, exit_reason=REASON[rc], bars_held=int(xi-ei)))
v2b = pd.concat([v2b, pd.DataFrame(res)], axis=1)
v2b['year'] = v2b.entry_date.dt.year
# (b) ruled out explicitly: no IS trade may have its 120-bar time stop clipped by the cut
_n_bars = v2b.ticker.map({tk: len(p['Close']) for tk, p in price_db.items()})
_clip = (v2b.entry_date <= PURGE_DATE) & (v2b.entry_idx + TIME_STOP_DAYS > _n_bars)
assert not _clip.any(), f'{int(_clip.sum())} IS trades have the time stop clipped by CUT_DATE'
# TIE ALIGNMENT for non-overlap. PIT_C's tr.sort_values(['ticker','entry_date']) is STABLE (lexsort),
# so among same-day duplicates on a ticker it kept the one earliest in its v2b FILE order. Rank our
# rows by that file order (IS rows only), then sort on an explicit 3-key so nothing rests on stability.
rf_path, rt_path = HIST_DIR/'cup_handle_v2b_fires_15yr_PIT_C.csv', HIST_DIR/'trades_t05_15yr_PIT_C.csv'
v2b['_ord'] = np.inf
if rf_path.exists():
    rf = with_tgt(read_is_only(rf_path, KEY[:-1] + ['target_price'])).drop(columns='target_price')
    rf['_ord_ref'] = np.arange(len(rf), dtype=float)
    v2b = v2b.merge(rf, on=KEY, how='left')
    v2b['_ord'] = v2b['_ord_ref'].fillna(np.inf); v2b = v2b.drop(columns='_ord_ref')
v2b = v2b.sort_values(['ticker','entry_date','_ord','lch_idx','rch_idx'], kind='mergesort').reset_index(drop=True)
# non-overlap per ticker (PIT_C rule), over every v2b fire <= CUT so IS rows see the same history
keep=[]; last={}
for r in v2b.itertuples():
    pv = last.get(r.ticker)
    if pv is None or r.entry_date > pv: keep.append(True); last[r.ticker] = r.exit_date
    else: keep.append(False)
v2b['kept_nonoverlap'] = keep
IS = v2b[v2b.entry_date <= PURGE_DATE].sort_values('entry_date').reset_index(drop=True)
del v2b, res                                   # nothing after PURGE_DATE goes any further
print(f'IS v2b fires (entry <= {PURGE_DATE.date()}): {len(IS)} | kept after non-overlap: {int(IS.kept_nonoverlap.sum())}')

# ---- parity vs the committed PIT_C run (IS rows only are ever read) ----
from collections import Counter
dup = IS.groupby(['ticker','entry_date']).size()
print(f'same-day duplicate cups in IS v2b: {int((dup > 1).sum())} (ticker, entry_date) groups, {int(dup[dup > 1].sum())} rows')
if rf_path.exists():
    # MULTISET on the full key — a set on (ticker, entry_date) hid the duplicate-cup divergence
    a = Counter(map(tuple, IS[KEY].values.tolist())); b = Counter(map(tuple, rf[KEY].values.tolist()))
    print(f'parity v2b fires (full key, multiset): mine {sum(a.values())} | PIT_C {sum(b.values())} '
          f'| only-mine {sum((a-b).values())} | only-PIT_C {sum((b-a).values())}')
    assert a == b, 'v2b fire multiset differs from PIT_C — stop and reconcile'
    assert np.isfinite(IS['_ord']).all() and IS['_ord'].is_unique, 'every IS row must carry a unique PIT_C file rank'
else:
    print('WARNING: PIT_C v2b fires file not found — parity NOT asserted')
if rt_path.exists():
    rt = read_is_only(rt_path, ['ticker','entry_date','stop_at_entry','R'])
    mine = IS[IS.kept_nonoverlap][['ticker','entry_date','stop_price','R']]
    assert not mine.duplicated(['ticker','entry_date']).any() and not rt.duplicated(['ticker','entry_date']).any()
    mm = mine.merge(rt, on=['ticker','entry_date'], how='outer', suffixes=('','_ref'), indicator=True)
    diff = (mm.R - mm.R_ref).abs()
    print(f'parity trades: matched {int((mm._merge=="both").sum())} | only-mine {int((mm._merge=="left_only").sum())} '
          f'| only-PIT_C {int((mm._merge=="right_only").sum())} | max |R diff| {diff.max():.2e}')
    if not ((mm._merge == 'both').all() and diff.max() < 1e-9):
        bad = mm[(mm._merge != 'both') | (diff >= 1e-9)]
        both = bad[bad._merge == 'both']
        print(f'  R mismatches among matched: {len(both)} | of those, stop differs: '
              f'{int(((both.stop_price - both.stop_at_entry).abs() > 1e-9).sum())} '
              f'(same stop + different R = a different left-rim cup with the same handle/entry)')
        print(f'  mismatched rows on a duplicate-cup day: '
              f'{int(bad.set_index(["ticker","entry_date"]).index.isin(dup[dup > 1].index).sum())} of {len(bad)}')
        print(bad.head(15).to_string(index=False))
    assert (mm._merge == 'both').all() and diff.max() < 1e-9, 'trade set / R differs from PIT_C — stop and reconcile'
else:
    print('WARNING: PIT_C trades file not found — parity NOT asserted')


In [ ]:
# ===== Cell 5 — Feature definitions (pure; every input is passed in) =====
FEATURES = ['spy_gt_200','spy_50_gt_200','breadth_50','rsp_spy_20d','rs_6m_pct','rs_12m_pct','px_52wh',
            'up_52wl_to_lch','up_6ml_to_lch','vol_bo_50','vol_handle_50','cup_depth_pct','cup_len',
            'handle_retr','handle_depth_atr','handle_len','handle_pos','gap_pct']
BINARY = {'spy_gt_200','spy_50_gt_200'}
TIMING = {f: ('entry' if f == 'gap_pct' else 'fire') for f in FEATURES}
nan = np.nan

def _idx_le(dates, d):
    """last index with date <= d (-1 if none)"""
    return int(np.searchsorted(dates, np.datetime64(d), side='right')) - 1

def mkt_features(d, spy, rsp):
    out = dict(spy_gt_200=nan, spy_50_gt_200=nan, rsp_spy_20d=nan)
    k = _idx_le(spy['Date'], d); c = spy['Close']
    if k >= 199:
        s200 = c[k-199:k+1].mean()
        out['spy_gt_200'] = float(c[k] > s200)
        out['spy_50_gt_200'] = float(c[k-49:k+1].mean() > s200)
    r = _idx_le(rsp['Date'], d)
    if k >= 20 and r >= 20:
        out['rsp_spy_20d'] = (rsp['Close'][r]/rsp['Close'][r-20] - 1) - (c[k]/c[k-20] - 1)
    return out

def stock_features(p, f):
    """f = the fire dict from detect_fires_full. Uses bars <= j = confirm_idx only."""
    j, lch, rch = int(f['confirm_idx']), int(f['lch_idx']), int(f['rch_idx'])
    H, L, C = p['High'], p['Low'], p['Close']
    V = np.where(p['Volume'] > 0, p['Volume'], nan)          # zero volume treated as missing
    out = {}
    out['px_52wh'] = C[j] / H[j-251:j+1].max() if j >= 251 else nan
    out['up_52wl_to_lch'] = H[lch] / L[lch-251:lch+1].min() - 1 if lch >= 251 else nan
    out['up_6ml_to_lch']  = H[lch] / L[lch-125:lch+1].min() - 1 if lch >= 125 else nan
    base = V[j-50:j] if j >= 50 else np.array([])
    avg50 = np.nanmean(base) if np.isfinite(base).sum() >= 40 else nan
    out['vol_bo_50'] = V[j] / avg50
    hv = V[rch+1:j]
    out['vol_handle_50'] = (np.nanmean(hv) / avg50) if np.isfinite(hv).any() else nan
    out['cup_depth_pct'] = float(f['cup_depth_pct'])
    out['cup_len'] = rch - lch
    out['handle_retr'] = float(f['handle_retr_pct'])
    out['handle_depth_atr'] = float(f['handle_depth_atr'])
    out['handle_len'] = j - rch
    out['handle_pos'] = (f['handle_low'] - f['cup_low']) / (f['cup_rim'] - f['cup_low'])
    return out

def gap_feature(p, f):
    """timing=entry: uses the entry-bar OPEN only."""
    return p['Open'][int(f['entry_idx'])] / p['Close'][int(f['confirm_idx'])] - 1

def build_panels(price_db, cal):
    """Universe panels on the SPY calendar. Row d depends only on rows <= d (rolling/shift/row-rank)."""
    close = pd.DataFrame({tk: pd.Series(p['Close'], index=pd.DatetimeIndex(p['Date'])) for tk, p in price_db.items()})
    close = close.reindex(pd.DatetimeIndex(cal))
    sma50 = close.rolling(50, min_periods=50).mean()
    elig = close.notna() & sma50.notna()
    breadth = ((close > sma50) & elig).sum(axis=1) / elig.sum(axis=1)
    rs6  = (close / close.shift(126) - 1).rank(axis=1, pct=True)
    rs12 = (close / close.shift(252) - 1).rank(axis=1, pct=True)
    return dict(close=close, breadth=breadth, rs6=rs6, rs12=rs12)

def panel_features_fast(panels, d, tk):
    d = pd.Timestamp(d)
    if d not in panels['breadth'].index:
        return dict(breadth_50=nan, rs_6m_pct=nan, rs_12m_pct=nan)
    g = lambda k: float(panels[k].at[d, tk]) if tk in panels[k].columns else nan
    return dict(breadth_50=float(panels['breadth'].at[d]), rs_6m_pct=g('rs6'), rs_12m_pct=g('rs12'))

def panel_features_slow(close_hist, d, tk):
    """Same definitions recomputed from scratch on the close panel TRUNCATED at d."""
    d = pd.Timestamp(d)
    if len(close_hist) == 0 or close_hist.index[-1] != d:
        return dict(breadth_50=nan, rs_6m_pct=nan, rs_12m_pct=nan)
    last = close_hist.iloc[-1]
    if len(close_hist) >= 50:
        w = close_hist.iloc[-50:]; ok = w.notna().all() & last.notna(); sma = w.mean()
        breadth = ((last > sma) & ok).sum() / ok.sum()
    else:
        breadth = nan
    def rs(n):
        if len(close_hist) <= n: return nan
        r = last / close_hist.iloc[-1-n] - 1
        return float(r.rank(pct=True).get(tk, nan))
    return dict(breadth_50=float(breadth), rs_6m_pct=rs(126), rs_12m_pct=rs(252))

def feature_row(f, price_db, spy, rsp, panels):
    p = price_db[f['ticker']]; d = pd.Timestamp(p['Date'][int(f['confirm_idx'])])
    r = dict(fire_date=d)
    r.update(mkt_features(d, spy, rsp)); r.update(panel_features_fast(panels, d, f['ticker']))
    r.update(stock_features(p, f)); r['gap_pct'] = gap_feature(p, f)
    return r


In [ ]:
# ===== Cell 6 — Feature table + CUT-OFF CAUSALITY TEST (aborts on any mismatch) =====
GEOM = ['lch_idx','rch_idx','h_idx','cup_low_idx','cup_low','cup_rim','handle_low','breakout_level',
        'cup_depth_pct','handle_retr_pct','handle_depth_atr','atr','entry_price','stop_price']

def build_feature_table(IS, price_db, spy, rsp, panels):
    rows = [feature_row(f, price_db, spy, rsp, panels) for f in IS.to_dict('records')]
    ids = IS[['ticker','entry_date','handle_low_date','confirm_idx','entry_idx','R','exit_reason',
              'exit_date','bars_held','year','kept_nonoverlap']].reset_index(drop=True)
    return pd.concat([ids, pd.DataFrame(rows)], axis=1)

def _trunc(p, last):
    return {k: (v[:last+1].copy() if isinstance(v, np.ndarray) else v) for k, v in p.items()}

def _rederive(q):
    """ATR / SMAs recomputed from the truncated raw bars only."""
    d = pd.DataFrame({'High': q['High'], 'Low': q['Low'], 'Close': q['Close']})
    q['ATR'] = compute_atr(d, ATR_PERIOD).to_numpy(float)
    q['S50'] = d['Close'].rolling(STOCK_FAST_SMA).mean().to_numpy(float)
    q['S200'] = d['Close'].rolling(STOCK_SLOW_SMA).mean().to_numpy(float)
    return q

def _same(a, b):
    a, b = float(a), float(b)
    if np.isnan(a) and np.isnan(b): return True
    return abs(a - b) <= 1e-9 * max(1.0, abs(a))

def _leaky_stock_features(p, f):
    """NEGATIVE CONTROL — peeks one bar past the fire bar. The tester must catch it."""
    out = stock_features(p, f); j = int(f['confirm_idx'])
    out['LEAK_next_close'] = p['Close'][j+1] / p['Close'][j] - 1 if len(p['Close']) > j+1 else nan
    return out

def cutoff_test(IS, price_db, spy, rsp, panels, n=200, seed=0, leak=False, verbose=True):
    """For sampled fires: every feature computed on FULL data must equal the same feature
    re-derived on data CUT at the fire bar j (fire-timed) or at j+1 with only its OPEN known
    (geometry re-detection + gap_pct). Returns the number of mismatches."""
    rng = np.random.default_rng(seed)
    pick = rng.choice(len(IS), size=min(n, len(IS)), replace=False)
    sfn = _leaky_stock_features if leak else stock_features
    bad = []
    for i in pick:
        f = IS.iloc[i].to_dict(); tk = f['ticker']; p = price_db[tk]
        j = int(f['confirm_idx']); e = j + 1; d = pd.Timestamp(p['Date'][j])
        # (B) bars through e, but only e's OPEN is known
        qB = _trunc(p, e)
        for k in ('High', 'Low', 'Close', 'Volume'): qB[k][e] = nan
        qB = _rederive(qB)
        # several cups (different left rims) can share one breakout bar -> match the exact cup
        g = [x for x in detect_fires_full(tk, qB, DET)
             if (x['confirm_idx'], x['lch_idx'], x['rch_idx']) == (j, int(f['lch_idx']), int(f['rch_idx']))]
        if len(g) != 1:
            bad.append((tk, str(d.date()), 'fire not re-detected at j+1 cut')); continue
        g = g[0]
        bad += [(tk, str(d.date()), f'geometry {k}') for k in GEOM if not _same(f[k], g[k])]
        if not _same(gap_feature(p, f), gap_feature(qB, g)): bad.append((tk, str(d.date()), 'gap_pct'))
        # (A) bars through j only — every fire-timed feature
        qA = _rederive(_trunc(p, j))
        full = sfn(p, f); cut = sfn(qA, g)
        k_s, k_r = _idx_le(spy['Date'], d), _idx_le(rsp['Date'], d)
        spyA = dict(Date=spy['Date'][:k_s+1], Close=spy['Close'][:k_s+1])
        rspA = dict(Date=rsp['Date'][:k_r+1], Close=rsp['Close'][:k_r+1])
        full.update(mkt_features(d, spy, rsp)); cut.update(mkt_features(d, spyA, rspA))
        full.update(panel_features_fast(panels, d, tk))
        cut.update(panel_features_slow(panels['close'].loc[:d], d, tk))
        bad += [(tk, str(d.date()), k) for k in full if not _same(full[k], cut[k])]
    if verbose:
        tag = 'NEGATIVE CONTROL (leaky feature injected)' if leak else 'cut-off causality test'
        print(f'{tag}: {len(pick)} fires, {len(bad)} mismatches' + (f' e.g. {bad[:5]}' if bad else ''))
    return len(bad)
# ---- run ----
t0 = time.time()
panels = build_panels(price_db, SPY['Date'])
FT = build_feature_table(IS, price_db, SPY, RSP, panels)
print(f'feature table: {len(FT)} rows x {len(FEATURES)} features ({time.time()-t0:.0f}s)')
print('non-NaN share per feature:', FT[FEATURES].notna().mean().round(3).to_dict())
n_ctrl = cutoff_test(IS, price_db, SPY, RSP, panels, n=20, seed=1, leak=True)
assert n_ctrl > 0, 'NEGATIVE CONTROL PASSED — the tester is blind; do not trust the test below'
n_bad = cutoff_test(IS, price_db, SPY, RSP, panels, n=200, seed=0)
assert n_bad == 0, 'CAUSALITY TEST FAILED — no tables until fixed'


In [ ]:
# ===== Cell 7 — PRE-REGISTRATION (written before any table; immutable once written) =====
PREREG = dict(
    sweep_id=SWEEP_ID, built_from_commit=BUILT_FROM_COMMIT,
    directions=dict(
        higher_is_better=['spy_gt_200','spy_50_gt_200','breadth_50','rsp_spy_20d','rs_6m_pct','rs_12m_pct',
                          'px_52wh','up_52wl_to_lch','up_6ml_to_lch','vol_bo_50','handle_pos'],
        lower_is_better=['vol_handle_50','gap_pct'],
        no_prior=['cup_depth_pct','cup_len','handle_retr','handle_depth_atr','handle_len']),
    criteria=CRITERIA, cost_R=COST_R,
    lockbox=dict(cut_date=str(CUT_DATE.date()), purge_date=str(PURGE_DATE.date()),
                 prior_exposure='PIT_C base and hscore quintiles already evaluated on 2025+ (PIT_C notebook Cell 7)'),
    rules=dict(continuous='IS-quintile buckets; B1..B5, GE2..GE5, LE1..LE4 (13 rules, both directions)',
               binary='EQ1, EQ0'),
    population='PIT_C v2b fires, kept after per-ticker non-overlap, entry <= purge date',
    features=FEATURES, timing=TIMING)
_dirs = PREREG['directions']
assert sorted(sum(_dirs.values(), [])) == sorted(FEATURES), 'prereg must cover every feature exactly once'
PREREG_PATH = OUT_DIR/'prereg_PIT_C.json'
if PREREG_PATH.exists():
    old = json.load(open(PREREG_PATH))
    for k in ['sweep_id','directions','criteria','cost_R','lockbox','rules','population','features']:
        assert old[k] == json.loads(json.dumps(PREREG[k])), f'prereg field {k!r} differs from the registered file — refusing to run'
    print('prereg_PIT_C.json already registered at', old['registered_at'], '— unchanged, OK')
else:
    PREREG['registered_at'] = pd.Timestamp.now(tz='UTC').isoformat()
    json.dump(PREREG, open(PREREG_PATH, 'w'), indent=1)
    print('registered', PREREG_PATH)
PREREG_DIR = {f: d for d, fs in (('higher', _dirs['higher_is_better']), ('lower', _dirs['lower_is_better']),
                                 ('none', _dirs['no_prior'])) for f in fs}


In [ ]:
# ===== Cell 8 — Single-factor tables (IS only) + trial log =====
def pf(x):
    x = np.asarray(x, float); w = x[x > 0].sum(); l = -x[x < 0].sum()
    return w / l if l > 0 else (np.inf if w > 0 else np.nan)

def month_block_ci(Rc, months, level, reps, seed):
    """PF CI by resampling whole calendar months (fires cluster in time)."""
    if len(Rc) == 0: return nan, nan
    um, inv = np.unique(months, return_inverse=True)
    W = np.bincount(inv, weights=np.clip(Rc, 0, None), minlength=len(um))
    Lo = np.bincount(inv, weights=np.clip(-Rc, 0, None), minlength=len(um))
    idx = np.random.default_rng(seed).integers(0, len(um), (reps, len(um)))
    w, l = W[idx].sum(1), Lo[idx].sum(1)
    pfs = np.where(l > 0, w / np.where(l > 0, l, 1), np.inf)
    a = (1 - level) / 2
    return float(np.quantile(pfs, a)), float(np.quantile(pfs, 1 - a))

def bucketize(x):
    edges = np.unique(np.nanquantile(x, [.2, .4, .6, .8]))
    b = np.where(np.isnan(x), 0, np.searchsorted(edges, x, side='right') + 1)
    return edges, b, len(edges) + 1

def rule_masks(b, k):
    m = {f'B{i}': b == i for i in range(1, k+1)}
    m.update({f'GE{i}': b >= i for i in range(2, k+1)})
    m.update({f'LE{i}': (b >= 1) & (b <= i) for i in range(1, k)})
    return m

def rule_direction(rule, k):
    if rule.startswith('GE') or rule == 'EQ1': return 'high'
    if rule.startswith('LE') or rule == 'EQ0': return 'low'
    i, mid = int(rule[1:]), (k + 1) / 2
    return 'high' if i > mid else ('low' if i < mid else 'mid')

def against_prereg(prereg_dir, rule_dir):
    return (prereg_dir == 'higher' and rule_dir == 'low') or (prereg_dir == 'lower' and rule_dir == 'high')

def config_id(feature, rule, edges):
    key = json.dumps([SWEEP_ID, feature, rule, [round(float(e), 10) for e in edges],
                      str(CUT_DATE.date()), str(PURGE_DATE.date()), COST_R, CRITERIA], sort_keys=True)
    return hashlib.sha1(key.encode()).hexdigest()[:12]

def eval_rule(mask, R, years, months, base_year, cid, crit=None):
    crit = crit or CRITERIA
    r = R[mask]; rc = r - COST_R; y = years[mask]; n = len(r)
    out = dict(n=n, n_months=len(np.unique(months[mask])) if n else 0)
    if n == 0:
        out.update(WR=nan, PF=nan, PF_cost=nan, avgR=nan, avgR_cost=nan, ci_lo=nan, ci_hi=nan,
                   years_beat=0, max_year_share=nan, mean_Rc=nan, std_Rc=nan, skew_Rc=nan, kurt_Rc=nan)
    else:
        lo, hi = month_block_ci(rc, months[mask], crit['ci_level'], crit['boot_reps'], int(cid[:8], 16))
        yr_net = pd.Series(rc).groupby(y).sum(); tot = rc.sum()
        sd = rc.std()
        out.update(WR=(r > 0).mean(), PF=pf(r), PF_cost=pf(rc), avgR=r.mean(), avgR_cost=rc.mean(),
                   ci_lo=lo, ci_hi=hi,
                   years_beat=int(sum(1 for yy, bm in base_year.items()
                                      if (y == yy).any() and rc[y == yy].mean() > bm)),
                   max_year_share=(yr_net.max() / tot) if tot > 0 else np.inf,
                   mean_Rc=rc.mean(), std_Rc=sd,
                   skew_Rc=((rc - rc.mean())**3).mean() / sd**3 if sd > 0 else nan,
                   kurt_Rc=((rc - rc.mean())**4).mean() / sd**4 if sd > 0 else nan)   # Pearson (non-excess)
    out['pass_pf'] = bool(out['PF_cost'] >= crit['min_pf_cost']) if n else False
    out['pass_ci'] = bool(out['ci_lo'] > crit['min_ci_lo']) if n else False
    out['pass_n'] = bool(n >= crit['min_n'])
    out['pass_years'] = bool(out['years_beat'] >= crit['min_years_beat'])
    out['pass_share'] = bool(n and np.isfinite(out['max_year_share']) and out['max_year_share'] <= crit['max_year_share'])
    out['PASS'] = all(out[k] for k in ['pass_pf', 'pass_ci', 'pass_n', 'pass_years', 'pass_share'])
    return out

def run_sweep(POP, features, binary, prereg_dir, crit=None):
    """Returns (trial rows, monthly-R rows, per-year rows). POP = IS analysis population."""
    R = POP['R'].to_numpy(float); years = POP['year'].to_numpy()
    months = POP['entry_date'].dt.to_period('M').astype(str).to_numpy()
    base_year = {yy: (R[years == yy] - COST_R).mean() for yy in np.unique(years)}
    trials, mrows, yrows = [], [], []
    def add(feature, rule, edges, mask, k, n_nan):
        cid = config_id(feature, rule, edges)
        st = eval_rule(mask, R, years, months, base_year, cid, crit)
        rd = 'base' if feature == 'BASELINE' else rule_direction(rule, k)
        pdir = prereg_dir.get(feature, 'n/a')
        trials.append(dict(kind='baseline' if feature == 'BASELINE' else 'trial', config_id=cid, sweep_id=SWEEP_ID,
                           feature=feature, rule=rule, rule_dir=rd, prereg_dir=pdir,
                           against_prereg=against_prereg(pdir, rd), timing=TIMING.get(feature, ''),
                           edges=json.dumps([round(float(e), 6) for e in edges]), n_nan=n_nan, **st))
        if mask.any():
            g = pd.DataFrame({'month': months[mask], 'R': R[mask]}).groupby('month')['R'].agg(['size', 'sum'])
            mrows.extend(dict(config_id=cid, month=mo, n=int(v['size']), sumR=float(v['sum'])) for mo, v in g.iterrows())
    add('BASELINE', 'ALL', [], np.ones(len(R), bool), 1, 0)
    for feat in features:
        x = POP[feat].to_numpy(float); n_nan = int(np.isnan(x).sum())
        if feat in binary:
            edges, k, masks = [0.5], 2, {'EQ1': x == 1, 'EQ0': x == 0}
            lo_m, hi_m = masks['EQ0'], masks['EQ1']
        else:
            edges, b, k = bucketize(x); masks = rule_masks(b, k)
            lo_m, hi_m = masks['B1'], masks[f'B{k}']
        for rule, m in masks.items(): add(feat, rule, edges, m, k, n_nan)
        for yy in np.unique(years):
            l, h = R[lo_m & (years == yy)], R[hi_m & (years == yy)]
            yrows.append(dict(feature=feat, year=int(yy), n_low=len(l), PF_low=pf(l), n_high=len(h), PF_high=pf(h),
                              avgR_spread=(h.mean() if len(h) else nan) - (l.mean() if len(l) else nan)))
    return pd.DataFrame(trials), pd.DataFrame(mrows), pd.DataFrame(yrows)

def append_log(path, df, key='config_id'):
    """Append-only; an identical config already logged is not a new trial."""
    if path.exists():
        old = pd.read_csv(path, dtype={key: str})
        new = df[~df[key].isin(old[key])]
        pd.concat([old, new], ignore_index=True).to_csv(path, index=False)
        return len(new), len(old) + len(new)
    df.to_csv(path, index=False); return len(df), len(df)

def exposure_rows():
    now = pd.Timestamp.now(tz='UTC').isoformat()
    return pd.DataFrame([dict(kind='exposure_note', config_id=f'exposure_{x}', sweep_id=SWEEP_ID, feature=x, rule='',
                              note='already evaluated on 2025+ (PIT_C notebook Cell 7): lockbox NOT clean for this', logged_at=now)
                         for x in ['PIT_C_base', 'hscore']])
# ---- run ----
POP = FT[FT.kept_nonoverlap].reset_index(drop=True)
assert POP.entry_date.max() <= PURGE_DATE
yrs = sorted(POP.year.unique())
print('PASS CRITERIA:', CRITERIA, '| cost', COST_R, 'R')
print(f'IS population: {len(POP)} trades | {len(yrs)} IS years ({yrs[0]}-{yrs[-1]})'
      + ('' if len(yrs) == CRITERIA['n_is_years_expected'] else f'  !! expected {CRITERIA["n_is_years_expected"]} years — years criterion stays >= {CRITERIA["min_years_beat"]}'))
trials, mret, yearly = run_sweep(POP, FEATURES, BINARY, PREREG_DIR)
trials['logged_at'] = pd.Timestamp.now(tz='UTC').isoformat(); trials['built_from_commit'] = BUILT_FROM_COMMIT

pd.set_option('display.width', 220)
fmt = lambda v: f'{v:.3f}'
b0 = trials[trials.kind == 'baseline'].iloc[0]
print(f"\nBASELINE: n={b0.n}  WR={b0.WR:.3f}  PF={b0.PF:.3f}  PF_cost={b0.PF_cost:.3f}  CI90=[{b0.ci_lo:.2f},{b0.ci_hi:.2f}]  avgR={b0.avgR:+.3f}")
cols = ['rule','n','n_months','WR','PF','PF_cost','ci_lo','avgR','years_beat','max_year_share','PASS']
for feat in FEATURES:
    t = trials[trials.feature == feat]
    print(f"\n=== {feat}  [prereg: {PREREG_DIR[feat]} | timing: {TIMING[feat]}]  edges {t.edges.iloc[0]}  NaN {t.n_nan.iloc[0]}")
    bk = t[t.rule.str.match(r'^(B\d|EQ\d)$')]
    print(bk[cols].to_string(index=False, float_format=fmt))
    y = yearly[yearly.feature == feat]
    print(f"  per-year (lowest vs highest bucket): high>low avgR in {int((y.avgR_spread > 0).sum())} of {len(y)} years")
    print(y[['year','n_low','PF_low','n_high','PF_high','avgR_spread']].to_string(index=False, float_format=fmt))

tr_rules = trials[trials.kind == 'trial']
passed = tr_rules[tr_rules.PASS]
print(f"\n=== RULES PASSING ALL CRITERIA: {len(passed)} of {len(tr_rules)} trials ===")
if len(passed):
    print(passed[['feature','rule','rule_dir','prereg_dir','against_prereg','n','PF_cost','ci_lo','years_beat','max_year_share']]
          .to_string(index=False, float_format=fmt))
    if passed.against_prereg.any():
        print('!! FLAG: passing rule(s) AGAINST the pre-registered direction:',
              passed[passed.against_prereg][['feature','rule']].values.tolist())
print('\nnear misses (fail exactly one criterion):')
flags = ['pass_pf','pass_ci','pass_n','pass_years','pass_share']
nm = tr_rules[(~tr_rules.PASS) & (tr_rules[flags].sum(axis=1) == 4)]
print(nm[['feature','rule','n','PF_cost','ci_lo','years_beat','max_year_share'] + flags].to_string(index=False, float_format=fmt) if len(nm) else '  none')

corr = POP[FEATURES].corr(method='spearman')
print('\nSpearman |rho| >= 0.6 pairs:')
for i, a in enumerate(FEATURES):
    for b in FEATURES[i+1:]:
        if abs(corr.at[a, b]) >= 0.6: print(f'  {a} ~ {b}: {corr.at[a, b]:+.2f}')


In [ ]:
# ===== Cell 9 — Save (IS only; new files only) =====
LOG, RET = OUT_DIR/'sweep_log_PIT_C.csv', OUT_DIR/'sweep_returns_PIT_C.csv'
if not LOG.exists(): exposure_rows().to_csv(LOG, index=False)
n_new, n_tot = append_log(LOG, trials)
r_new, r_tot = append_log(RET, mret.assign(key=mret.config_id + '|' + mret.month), key='key')
log_all = pd.read_csv(LOG)
print(f'trial log: +{n_new} new configs this run | cumulative trials (kind=trial): {int((log_all.kind=="trial").sum())}')
print(f'monthly-R rows: +{r_new} (total {r_tot})')
FT.to_csv(OUT_DIR/'features_IS_PIT_C.csv', index=False)
trials.to_csv(OUT_DIR/'factor_tables_IS_PIT_C.csv', index=False)
yearly.to_csv(OUT_DIR/'factor_per_year_IS_PIT_C.csv', index=False)
corr.to_csv(OUT_DIR/'feature_corr_IS_PIT_C.csv')
print('saved to', OUT_DIR, ':', sorted(p.name for p in OUT_DIR.iterdir()))
